## Setup

In [2]:
import pandas as pd
from functools import reduce

In [4]:
# Step 1: Define paths to your new Merton census files
# This assumes the new files are in the same directory as your notebook
merton_paths = {
    'edu_2021': '/Users/callumhall/Downloads/Merton_Quals.csv',
    'tenure_2021': '/Users/callumhall/Downloads/Merton_Tenure.csv',
    'occ_2021': '/Users/callumhall/Downloads/Merton_Occ.csv',
    'age_2021': '/Users/callumhall/Downloads/Merton_Age.csv',
    'eth_2021': '/Users/callumhall/Downloads/Merton_Eth.csv',
    'property_2021': '/Users/callumhall/Downloads/Merton_Property.csv'
}

In [6]:
# Step 2: Define the column mappings (these are the same as your main script)
edu_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All usual residents aged 16 years and over': 'Total_Population', 'No qualifications': 'Edu_No_Quals', 'Level 1 and entry level qualifications': 'Edu_Level_1_Quals', 'Level 2 qualifications': 'Edu_Level_2_Quals', 'Apprenticeship': 'Edu_Apprentice', 'Level 3 qualifications': 'Edu_Level_3_Quals', 'Level 4 qualifications or above': 'Edu_Level_4_Plus'}
tenure_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All households': 'Total_Households', 'Owned': 'Tenure_Owned', 'Social rented': 'Tenure_Social_Rented','Private rented': 'Tenure_Private_Rented'}
occ_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All usual residents aged 16 years and over in employment the week before the census': 'Total_Employed', '1. Managers, directors and senior officials': 'Occ_Managers', '2. Professional occupations': 'Occ_Professional', '3. Associate professional and technical occupations': 'Occ_Associate_Professional', '5. Skilled trades occupations': 'Occ_Skilled_Trades'}
age_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total': 'Total_Age_Population','Aged 15 to 19 years': 'Age_15_19','Aged 20 to 24 years': 'Age_20_24', 'Aged 25 to 29 years': 'Age_25_29', 'Aged 30 to 34 years': 'Age_30_34', 'Aged 35 to 39 years': 'Age_35_39', 'Aged 40 to 44 years': 'Age_40_44', 'Aged 45 to 49 years': 'Age_45_49', 'Aged 50 to 54 years': 'Age_50_54', 'Aged 55 to 59 years': 'Age_55_59', 'Aged 60 to 64 years': 'Age_60_64'}
eth_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name', 'Total: All usual residents': 'Total_Ethnic_Population', 'White': 'Ethnic_White', 'Mixed or Multiple ethnic groups': 'Ethnic_Mixed', 'Asian, Asian British or Asian Welsh': 'Ethnic_Asian', 'Black, Black British, Black Welsh, Caribbean or African': 'Ethnic_Black'}
property_cols_21 = {'2021 super output area - lower layer': 'LSOA_Code_Name','Total': 'Total_Properties', 'Detached': 'Property_Detached', 'Semi-detached': 'Property_Semi', 'Terraced': 'Property_Terraced', 'In a purpose-built block of flats or tenement': 'Flats'}

skip_values_21 = {'edu_2021': 7, 'tenure_2021': 7, 'occ_2021': 7, 'age_2021': 5, 'eth_2021': 7, 'property_2021': 7}
col_maps_21 = {'edu_2021': edu_cols_21, 'tenure_2021': tenure_cols_21, 'occ_2021': occ_cols_21, 'age_2021': age_cols_21, 'eth_2021': eth_cols_21, 'property_2021': property_cols_21}

In [8]:
# Step 3: Reusable Cleaning Function (from your preprocessing notebook)
def load_and_clean_census_file(path, column_map, skiprows):
    try:
        df = pd.read_csv(path, skiprows=skiprows, usecols=list(column_map.keys()), encoding='ISO-8859-1')
        df = df.rename(columns=column_map)
        df[['LSOA_Code', 'LSOA_Name']] = df['LSOA_Code_Name'].str.split(' : ', n=1, expand=True)
        df = df.drop(columns=['LSOA_Code_Name', 'LSOA_Name'])
        df.dropna(subset=['LSOA_Code'], inplace=True)
        data_cols = [col for col in df.columns if col != 'LSOA_Code']
        for col in data_cols:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
        return df
    except Exception as e:
        print(f"Error processing file {path}: {e}")
        return pd.DataFrame()

## Appending Merton Data

In [12]:
# Step 1: Process and merge the five new Merton files
print("--- Processing new Merton data files ---")
dfs_merton = [load_and_clean_census_file(merton_paths[name], col_maps_21[name], skip_values_21[name]) for name in col_maps_21]
df_merton_final = reduce(lambda left, right: pd.merge(left, right, on='LSOA_Code', how='inner'), [df for df in dfs_merton if not df.empty])
print(f"Cleaned Merton data has {len(df_merton_final)} rows (LSOAs).")

--- Processing new Merton data files ---
Cleaned Merton data has 126 rows (LSOAs).


In [20]:
# Step 2: Load your main cleaned census file
main_census_path = '/Users/callumhall/Documents/Dissertation/Datasets/Cleaned Data/cleaned_census.csv'
df_main_census = pd.read_csv(main_census_path)
print(f"Original cleaned census file has {len(df_main_census)} rows.")

Original cleaned census file has 4862 rows.


In [22]:
# Step 3: Append the new Merton data
df_updated_census = pd.concat([df_main_census, df_merton_final], ignore_index=True)
print(f"New combined census file has {len(df_updated_census)} rows.")

New combined census file has 4988 rows.


In [24]:
# Step 4: Save the updated file, overwriting the old one
df_updated_census.to_csv(main_census_path, index=False)
print(f"\nSuccessfully added Merton data to '{main_census_path}'.")


Successfully added Merton data to '/Users/callumhall/Documents/Dissertation/Datasets/Cleaned Data/cleaned_census.csv'.
